In [69]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [70]:
import os
os.chdir(r'C:\Kaggle_Competition\Playground\S6E5-F1-Pit-Stops')
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from scipy.stats import rankdata

# Externals
from src.utils import *
import config
import warnings
warnings.filterwarnings('ignore')

In [71]:
train = read_csv_file('data/raw/train.csv')
test = read_csv_file('data/raw/test.csv')

train = reduce_mem_usage(train)
test = reduce_mem_usage(test)

train_ids = train[config.ID_COL].values
test_ids = test[config.ID_COL].values

print(f'Shape of train data: {train.shape}')
print(f'Shape of test data: {test.shape}')

# Data
X = train.drop(['id', config.TARGET], axis=1)
y = train[config.TARGET].values
X_test = test.drop('id', axis=1)

Shape of train data: (439140, 16)
Shape of test data: (188165, 15)


### Weighted Blend

In [68]:
MODELS = [
    ("HistGBM", r"artifacts\oof\HISTGBM_v2_oof_proba.csv", r"artifacts\test_proba\HISTGBM_v2_test_proba.csv"),
    ("XGBM", r"artifacts\oof\XGB_v13_seed42_oof_proba.csv", r"artifacts\test_proba\XGB_v13_seed42_test_proba.csv"),
    ("LGBM", r"artifacts\oof\LGBM_v4_seed42_oof_proba.csv", r"artifacts\test_proba\LGBM_v4_seed42_test_proba.csv"),
    ("ResNet", r"artifacts\oof\RESNET_v1_seed42_oof_proba.csv", r"artifacts\test_proba\RESNET_v1_seed42_test_proba.csv"),
    ("REALMLP", r"artifacts\oof\REALMLP_v2_seed42_oof_proba.csv", r"artifacts\test_proba\REALMLP_v2_seed42_test_proba.csv"),
    ("CatBoost", r"artifacts\oof\CatGBM_v00_oof_proba.csv", r"artifacts\test_proba\CatGBM_v00_test_proba.csv"),
    ("TABM", r"artifacts\oof\TABM_v1_seed42_oof_proba.csv", r"artifacts\test_proba\TABM_v1_seed42_test_proba.csv"),
    ("STACK", r"artifacts\oof\STACK_v0_oof_proba.csv", r"artifacts\test_proba\STACK_v0_test_proba.csv"),
    ("STACK_LR", r"artifacts\oof\stack_lr_seedavg_v1_oof_proba.csv", r"artifacts\test_proba\stack_lr_seedavg_v1_test_proba.csv"),
    ("TABPFN", r"artifacts\oof\TAB-PFN_v2_seed42_oof_proba.csv", r"artifacts\test_proba\TAB-PFN_v2_seed42_test_proba.csv"),
    ('Blend_ext', r'artifacts\oof\blend_external_oof_proba.csv', r'artifacts\test_proba\blend_external_test_proba.csv')
]

def load_pred(path: str, pred_col: str = "PitNextLap") -> np.ndarray:

    df = read_csv_file(path)

    if pred_col in df.columns:
        return df[pred_col].to_numpy(dtype=np.float32)

    return df.iloc[:, 1].to_numpy(dtype=np.float32)


# Numpy matrices
names = [m[0] for m in MODELS]

oof_mat = np.column_stack([
    load_pred(m[1]) for m in MODELS
]).astype(np.float32)

test_mat = np.column_stack([
    load_pred(m[2]) for m in MODELS
]).astype(np.float32)


single_scores = {
    name: roc_auc_score(y, oof_mat[:, i])
    for i, name in enumerate(names)
}

for name, score in single_scores.items():
    print(f"{name} alone: {score:.6f}")


# Objective function
def objective(trial):

    raw = np.array(
        [
            trial.suggest_float(f"w_{name}", -3.0, 3.0)
            for name in names
        ],
        dtype=np.float32
    )

    # softmax normalization
    weights = np.exp(raw - raw.max())
    weights /= weights.sum()

    preds = oof_mat @ weights

    return roc_auc_score(y, preds)

# Phase 1 -> Quasi Monte Carlo
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.QMCSampler(
        seed=config.SEED,
        qmc_type='sobol',
        scramble=True,
        independent_sampler=optuna.samplers.RandomSampler(seed=config.SEED),
        warn_asynchronous_seeding=True,
        warn_independent_sampling=True
    ),
)

study.optimize(
    objective,
    n_trials=1000,
    show_progress_bar=True,
    n_jobs=1 # no parallelization to preserve descrepancy
)

# Phase 2 -> Tree-Prazen Estimator
study.sampler = optuna.samplers.TPESampler(
    n_ei_candidates=64,
    seed=config.SEED,
    multivariate=False,
    warn_independent_sampling=True,
    n_startup_trials=0,
    constant_liar=True # inject fake constants
)

study.optimize(
    objective,
    n_trials=5000,
    show_progress_bar=True,
    n_jobs=-1
)

# Best weights
best_raw = np.array(
    [
        study.best_params[f"w_{name}"]
        for name in names
    ],
    dtype=np.float32
)

best_w = np.exp(best_raw - best_raw.max())
best_w /= best_w.sum()

# Final weighted blend
blend_oof = oof_mat @ best_w
blend_test = test_mat @ best_w

# Results
best_single = max(single_scores.values())

print(f"\n{'='*50}")

print(f"Best blend: {study.best_value:.6f}")
print(f"Gain: +{study.best_value - best_single:.6f}")

print(f"\n{'='*50}")

for name, w in zip(names, best_w):
    print(f"{name} weight: {w:.4f} ({w:.1%})")

HistGBM alone: 0.945228
XGBM alone: 0.950772
LGBM alone: 0.948235
ResNet alone: 0.942910
REALMLP alone: 0.954065
CatBoost alone: 0.952983
TABM alone: 0.946562
STACK alone: 0.954471
STACK_LR alone: 0.954110
TABPFN alone: 0.949509
Blend_ext alone: 0.950153


  0%|          | 0/1000 [00:00<?, ?it/s]

[W 2026-05-20 23:17:12,430] Trial 50 failed with parameters: {'w_HistGBM': 0.18231491558253765, 'w_XGBM': -0.22292931005358696, 'w_LGBM': 0.37373495288193226, 'w_ResNet': 1.301647150889039, 'w_REALMLP': 2.9191743582487106, 'w_CatBoost': -0.2785752173513174, 'w_TABM': -0.8740331009030342, 'w_STACK': -0.8237980026751757, 'w_STACK_LR': 1.7476710565388203, 'w_TABPFN': 2.825814815238118, 'w_Blend_ext': -2.8715148959308863} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\aryan\miniconda3\envs\kaggle\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\aryan\AppData\Local\Temp\ipykernel_22068\1970981718.py", line 63, in objective
    return roc_auc_score(y, preds)
           ^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\aryan\miniconda3\envs\kaggle\Lib\site-packages\sklearn\utils\_param_validation.py", line 216, in wrapper
    return func(*a

KeyboardInterrupt: 

In [123]:
# Saving files
RUN_NAME = "JustAnotherBlend_v1[0.95489]"
blend_oof_df = pd.DataFrame({"id": train_ids, "PitNextLap": blend_oof})
blend_test_df = pd.DataFrame({"id": test_ids, "PitNextLap": blend_test})

save_csv_file(
    blend_oof_df,
    config.OOF_PROBA_DIR / f"{RUN_NAME}_oof_proba.csv"
)

save_csv_file(
    blend_test_df,
    config.TEST_PROBA_DIR / f"{RUN_NAME}_test_proba.csv"
)

In [ ]:
## best till now -> 0.95489

In [125]:
# Sanity check
sanity_oof = read_csv_file(r'artifacts\oof\JustAnotherBlend_v1[0.95489]_oof_proba.csv')
print(roc_auc_score(y, sanity_oof.iloc[:, 1]))

0.9548898524388371


### Weighted Ranked Blend

In [23]:
def objective(trial):
    w_hist = trial.suggest_float("w_hist", 0.0, 1.0)
    w_xgbm = trial.suggest_float("w_xgbm", 0.0, 1.0)
    w_lgbm = trial.suggest_float("w_lgbm", 0.0, 1.0)
    w_resnet = trial.suggest_float("w_resnet", 0.0, 1.0)
    w_realmlp = trial.suggest_float("w_realmlp", 0.0, 1.0)
    w_catboost = trial.suggest_float("w_catboost", 0.0, 1.0)

    total = (
        w_hist +
        w_xgbm +
        w_lgbm +
        w_resnet +
        w_realmlp +
        w_catboost
    )

    # Normalize weights [0, 1]
    w_hist /= total
    w_xgbm /= total
    w_lgbm /= total
    w_resnet /= total
    w_realmlp /= total
    w_catboost /= total

    preds = (
        w_hist * rankdata(oof_hist.ravel()) +
        w_xgbm * rankdata(oof_xgbm.ravel()) +
        w_lgbm * rankdata(oof_lgbm.ravel()) +
        w_resnet * rankdata(oof_resnet.ravel()) +
        w_realmlp * rankdata(oof_realmlp.ravel()) +
        w_catboost * rankdata(oof_catboost.ravel())
    )

    return roc_auc_score(target, preds)


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42)
)

study.optimize(
    objective,
    n_trials=5000,
    show_progress_bar=True
)

# -------- BEST WEIGHTS --------
best = study.best_params

total = (
    best["w_hist"] +
    best["w_xgbm"] +
    best["w_lgbm"] +
    best["w_resnet"] +
    best["w_realmlp"] +
    best["w_catboost"]
)

w_hist = best["w_hist"] / total
w_xgbm = best["w_xgbm"] / total
w_lgbm = best["w_lgbm"] / total
w_resnet = best["w_resnet"] / total
w_realmlp = best["w_realmlp"] / total
w_catboost = best["w_catboost"] / total

# -------- FINAL RANKED BLEND --------
blend_oof = (
    w_hist * rankdata(oof_hist.ravel()) +
    w_xgbm * rankdata(oof_xgbm.ravel()) +
    w_lgbm * rankdata(oof_lgbm.ravel()) +
    w_resnet * rankdata(oof_resnet.ravel()) +
    w_realmlp * rankdata(oof_realmlp.ravel()) +
    w_catboost * rankdata(oof_catboost.ravel())
)

blend_test = (
    w_hist * rankdata(test_hist.ravel()) +
    w_xgbm * rankdata(test_xgbm.ravel()) +
    w_lgbm * rankdata(test_lgbm.ravel()) +
    w_resnet * rankdata(test_resnet.ravel()) +
    w_realmlp * rankdata(test_realmlp.ravel()) +
    w_catboost * rankdata(test_catboost.ravel())
)

# -------- SCORES --------
best_single = max(
    score_hist,
    score_xgbm,
    score_lgbm,
    score_resnet,
    score_realmlp,
    score_catboost
)

print(f"\n{'='*40}")
print(f"HISTGBM alone: {score_hist:.6f}")
print(f"XGBM alone: {score_xgbm:.6f}")
print(f"LGBM alone: {score_lgbm:.6f}")
print(f"RESNET alone: {score_resnet:.6f}")
print(f"REALMLP alone: {score_realmlp:.6f}")
print(f"CATBOOST alone: {score_catboost:.6f}")

print(f"\nBest Ranked Blend: {study.best_value:.6f}")
print(f"Gain: +{study.best_value - best_single:.6f}")
print(f"{'='*40}")

print(f"HIST weight: {w_hist:.4f} ({w_hist:.1%})")
print(f"XGBM weight: {w_xgbm:.4f} ({w_xgbm:.1%})")
print(f"LGBM weight: {w_lgbm:.4f} ({w_lgbm:.1%})")
print(f"RESNET weight: {w_resnet:.4f} ({w_resnet:.1%})")
print(f"REALMLP weight: {w_realmlp:.4f} ({w_realmlp:.1%})")
print(f"CATBOOST weight: {w_catboost:.4f} ({w_catboost:.1%})")

  0%|          | 0/5000 [00:00<?, ?it/s]


HISTGBM alone: 0.945228
XGBM alone: 0.950772
LGBM alone: 0.948235
RESNET alone: 0.942910
REALMLP alone: 0.954034
CATBOOST alone: 0.952650

Best Ranked Blend: 0.954542
Gain: +0.000508
HIST weight: 0.0000 (0.0%)
XGBM weight: 0.0949 (9.5%)
LGBM weight: 0.0291 (2.9%)
RESNET weight: 0.0005 (0.0%)
REALMLP weight: 0.6312 (63.1%)
CATBOOST weight: 0.2443 (24.4%)


### Simple ranked average

In [24]:
# OOF ranked average
blend_oof = (
    rankdata(oof_hist.ravel()) +
    rankdata(oof_xgbm.ravel()) +
    rankdata(oof_lgbm.ravel()) +
    rankdata(oof_resnet.ravel()) +
    rankdata(oof_realmlp.ravel()) +
    rankdata(oof_catboost.ravel())
) / 6

# TEST ranked average
blend_test = (
    rankdata(test_hist.ravel()) +
    rankdata(test_xgbm.ravel()) +
    rankdata(test_lgbm.ravel()) +
    rankdata(test_resnet.ravel()) +
    rankdata(test_realmlp.ravel()) +
    rankdata(test_catboost.ravel())
) / 6

# Score
score = roc_auc_score(target, blend_oof)

print(f"Ranked Blend AUC: {score:.6f}")

Ranked Blend AUC: 0.952216


### Stacking

In [ ]:
oof_files = sorted(config.OOF_PROBA_DIR.glob("*_oof_proba.csv"))
test_files = sorted(config.TEST_PROBA_DIR.glob("*_test_proba.csv"))

oof_df = pd.DataFrame({"id": train_ids})
test_df = pd.DataFrame({"id": test_ids})

for i, file in enumerate(oof_files, start=1):
    oof_df[f"model_{i}"] = read_csv_file(file).iloc[:, 1].values

for i, file in enumerate(test_files, start=1):
    test_df[f"model_{i}"] = read_csv_file(file).iloc[:, 1].values

FEATURES = [col for col in oof_df.columns if col != "id"]

X_meta = oof_df[FEATURES].values
X_meta_test = test_df[FEATURES].values
y_meta = np.asarray(y)

In [14]:
print(X_meta.shape)
print(X_meta_test.shape)
print(y_meta.shape)

(439140, 37)
(188165, 37)
(439140,)


In [15]:
def objective(trial):

    selected_cols = [
        col for col in FEATURES
        if trial.suggest_int(f"use_{col}", 0, 1)
    ]

    if len(selected_cols) == 0:
        return 0.5

    X_selected = oof_df[selected_cols].values

    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    scores = []

    for fold, (tr_idx, val_idx) in enumerate(
        skf.split(X_selected, y_meta)
    ):

        X_tr = X_selected[tr_idx]
        X_val = X_selected[val_idx]

        y_tr = y_meta[tr_idx]
        y_val = y_meta[val_idx]

        model = make_pipeline(
            StandardScaler(),
            LogisticRegression(max_iter=5000)
        )

        model.fit(X_tr, y_tr)

        val_preds = model.predict_proba(X_val)[:, 1]

        fold_auc = roc_auc_score(y_val, val_preds)

        scores.append(fold_auc)

        # Pruning
        trial.report(np.mean(scores), step=fold)

        if trial.should_prune():
            raise optuna.TrialPruned()

    return np.mean(scores)


study = optuna.create_study(
    study_name='MY_STACK_V1',
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=10)
)

study.optimize(
    objective,
    n_trials=500,
    show_progress_bar=True,
    n_jobs=-1
)

[I 2026-05-16 02:09:23,717] A new study created in memory with name: MY_STACK_V1


  0%|          | 0/500 [00:00<?, ?it/s]

[I 2026-05-16 02:09:38,396] Trial 9 finished with value: 0.9534433914461374 and parameters: {'use_model_1': 0, 'use_model_2': 0, 'use_model_3': 0, 'use_model_4': 1, 'use_model_5': 1, 'use_model_6': 0, 'use_model_7': 1, 'use_model_8': 1, 'use_model_9': 0, 'use_model_10': 1, 'use_model_11': 1, 'use_model_12': 0, 'use_model_13': 1, 'use_model_14': 0, 'use_model_15': 1, 'use_model_16': 1, 'use_model_17': 0, 'use_model_18': 0, 'use_model_19': 0, 'use_model_20': 0, 'use_model_21': 0, 'use_model_22': 1, 'use_model_23': 0, 'use_model_24': 0, 'use_model_25': 1, 'use_model_26': 0, 'use_model_27': 0, 'use_model_28': 1, 'use_model_29': 0, 'use_model_30': 0, 'use_model_31': 0, 'use_model_32': 0, 'use_model_33': 0, 'use_model_34': 0, 'use_model_35': 0, 'use_model_36': 1, 'use_model_37': 0}. Best is trial 9 with value: 0.9534433914461374.
[I 2026-05-16 02:09:39,253] Trial 11 finished with value: 0.9535077725871256 and parameters: {'use_model_1': 0, 'use_model_2': 0, 'use_model_3': 1, 'use_model_4': 0

In [18]:
# best models
best_cols = [
    col for col in FEATURES
    if study.best_params[f"use_{col}"] == 1
]

In [22]:
best_cols = ['model_1', 'model_2', 'model_10', 'model_11', 'model_13', 'model_14', 'model_19', 'model_20', 'model_22', 'model_23', 'model_26', 'model_27', 'model_28', 'model_29', 'model_30', 'model_32', 'model_34', 'model_35', 'model_36', 'model_37']

In [23]:
X_best = oof_df[best_cols].values
X_best_test = test_df[best_cols].values
y_meta = np.asarray(y)

oof_seeds = []
test_seeds = []

for seed in config.SEED_LIST:
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)

    oof_seed = np.zeros(len(X_best))
    test_seed = np.zeros(len(X_best_test))

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_best, y_meta), start=1):
        X_tr, X_val = X_best[tr_idx], X_best[val_idx]
        y_tr, y_val = y_meta[tr_idx], y_meta[val_idx]

        model = make_pipeline(
            StandardScaler(),
            LogisticRegression(max_iter=5000)
        )

        model.fit(X_tr, y_tr)

        val_preds = model.predict_proba(X_val)[:, 1]
        tst_preds = model.predict_proba(X_best_test)[:, 1]

        oof_seed[val_idx] = val_preds
        test_seed += tst_preds / 5

    seed_auc = roc_auc_score(y_meta, oof_seed)
    print(f"Seed {seed} AUC: {seed_auc:.6f}")

    oof_seeds.append(oof_seed)
    test_seeds.append(test_seed)

oof_final = np.mean(oof_seeds, axis=0)
test_final = np.mean(test_seeds, axis=0)

final_auc = roc_auc_score(y_meta, oof_final)
print(f"\nFinal averaged AUC: {final_auc:.6f}")

oof_df_out = pd.DataFrame({"id": train_ids, "PitNextLap": oof_final})
test_df_out = pd.DataFrame({"id": test_ids, "PitNextLap": test_final})

save_csv_file(oof_df_out, config.OOF_PROBA_DIR / "stack_lr_seedavg_oof.csv")
save_csv_file(test_df_out, config.TEST_PROBA_DIR / "stack_lr_seedavg_test.csv")

Seed 42 AUC: 0.954095
Seed 71 AUC: 0.954104
Seed 84 AUC: 0.954113
Seed 69 AUC: 0.954101
Seed 91 AUC: 0.954097

Final averaged AUC: 0.954110
